In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage
from langgraph.graph.message import add_messages  ## Optimized reducer with BaseMessage Class
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from my_llm import *
from langchain_core.runnables import RunnableConfig
from langchain.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition 
import sqlite3
from tools import calculate, search_tool, get_stock_price

In [3]:
class ChatState(TypedDict): # Inherits from the TypedDict Class
    messages: Annotated[list[BaseMessage], add_messages]


In [4]:
def ChatNode(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state['messages']
    
    response = model_with_tools.invoke(messages)
    
    
    return {'messages': [response]}

In [5]:
tools = [calculate, search_tool, get_stock_price]
tool_node = ToolNode(tools)
model_with_tools = model.bind_tools(tools)

In [6]:
calculate.invoke({"a": 15, "b": 7, "operation": "multiply"})

105

In [7]:
graph = StateGraph(ChatState)

# Add nodes
graph.add_node("ChatNode",ChatNode)
graph.add_node("tools",tool_node)

## Connect
graph.set_entry_point("ChatNode")
graph.add_conditional_edges("ChatNode", tools_condition)
graph.add_edge("tools","ChatNode")

In [8]:
workflow = graph.compile()

In [44]:
# config: RunnableConfig = {"configurable": {"thread_id": 'thread-1'}}
response = workflow.invoke({'messages':[HumanMessage(content='what is the age of virat kohli considering his birth year and today date')]})

In [43]:
response

{'messages': [HumanMessage(content='what is the age of virat kohli considering his birth year and today date', additional_kwargs={}, response_metadata={}, id='7b5bd46c-a331-4965-90f7-7cd5b5a3b599'),
  AIMessage(content='Virat Kohli was born on November 5, 1988. Given that today is October 29, 2023, he has not yet reached his 35th birthday this year. Therefore, he is 34 years old.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 509, 'prompt_tokens': 194, 'total_tokens': 703, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'o3-mini-2025-01-31', 'system_fingerprint': 'fp_0d2bf52fac', 'id': 'chatcmpl-DLPyDdsuJQhyGU8j2CDkyyn56kcdg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {}}, id='lc_run--01

In [41]:
response["messages"][-1].content

'The biggest company in the world is often considered to be Apple, whose CEO is Tim Cook. Based on the latest available closing stock price of approximately 248.96 USD, multiplying this price by 100 gives roughly 24,896 USD.'

In [2]:
calculate.description

"Can perform calculations. Takes two integers as input and returns multiplication, division, addition, and subtraction of the two numbers.\n    use 'add' for addition, 'subtract' for subtraction, 'multiply' for multiplication, and 'divide' for division."